### Step R1.0 - Import CSV & Linear Combination Check


In [ ]:
suppressPackageStartupMessages({
  suppressWarnings({
    library(caret)
  })
})

### Read dataset
path <- "model_data.csv"
df <- read.csv(path, stringsAsFactors = FALSE)

In [3]:
head(df, 2)

,perc.inv.51.75,perc.reg.lt.25,perc.reg.26.50,perc.reg.51.75,env.little,env.some,env.very,info.little,info.some,info.very,⋯,imp.very,safe.little,safe.some,safe.very,max.lt.1,max.2_2.5,max.gt.2.5,y.ipo,y.real,perc.inv.lt.26.50
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,1,0,1,0,1,0,0,1,0,0,⋯,0,0,1,0,0,0,1,0,0,0
2,0,0,0,0,0,0,1,0,0,0,⋯,0,0,1,0,0,1,0,0,1,1


In [4]:
### Keep only regressors (exclude targets)
X <- df[ , !(names(df) %in% c("y.ipo","y.real")), drop = FALSE]

### Find linear combinations
res <- findLinearCombos(as.matrix(X))

if (length(res$linearCombos) == 0) {
  cat("No linear combinations found.\n")
} else {
  cat("Linear combinations detected (index sets):\n")
  print(res$linearCombos)
  cat("\nSuggested columns to remove:\n")
  cat(paste(names(X)[res$remove], collapse = ", "), "\n")
}


Linear combinations detected (index sets):
[[1]]
[1] 10  2  3  4  5  6

[[2]]
[1] 14  2  3  4  5  6  9

[[3]]
[1] 17  1  7  8  9 11 13 16

[[4]]
 [1] 18  1  2  4  5  6  7  8  9 11 12 13 15 16

[[5]]
[1] 19  1  7  8 11 13 16

[[6]]
 [1] 20  1  2  4  5  6  7  8  9 12 13 15 16

[[7]]
 [1] 21  1  2  4  5  6  7  8  9 11 12 13 15 16


Suggested columns to remove:
info.very, imp.very, safe.very, max.lt.1, max.2_2.5, max.gt.2.5, perc.inv.lt.26.50 


In [5]:
### Remove columns by index
X_clean <- X[, -c(10,14,17,18,19,20,21), drop = FALSE]

### Show new dimensions
cat("Clean regressors matrix shape:", dim(X_clean))


Clean regressors matrix shape: 97 14

### Step R2.0 - Classic Logit with X_clean on y.ipo and y.real


In [6]:
### Rebuild reduced dataset
df_clean <- data.frame(
  y.ipo  = df$y.ipo,
  y.real = df$y.real,
  X_clean,
  check.names = FALSE
)

### Logit for y.ipo
fit_yipo <- glm(y.ipo ~ ., data = df_clean, family = binomial(), na.action = na.omit)
cat("\n=== Classic Logit — y.ipo ===\n")
print(summary(fit_yipo))

### Logit for y.real (only if present)
if ("y.real" %in% names(df_clean)) {
  fit_yreal <- glm(y.real ~ ., data = df_clean, family = binomial(), na.action = na.omit)
  cat("\n=== Classic Logit — y.real ===\n")
  print(summary(fit_yreal))
}



=== Classic Logit — y.ipo ===

Call:
glm(formula = y.ipo ~ ., family = binomial(), data = df_clean, 
    na.action = na.omit)

Coefficients: (1 not defined because of singularities)
                 Estimate Std. Error z value Pr(>|z|)
(Intercept)    -2.657e+01  1.593e+05       0        1
y.real          5.313e+01  5.779e+05       0        1
perc.inv.51.75  5.313e+01  6.169e+05       0        1
perc.reg.lt.25  1.063e+02  5.263e+05       0        1
perc.reg.26.50  5.313e+01  2.085e+05       0        1
perc.reg.51.75  2.125e+02  8.967e+05       0        1
env.little     -1.063e+02  8.470e+05       0        1
env.some       -1.594e+02  1.053e+06       0        1
env.very               NA         NA      NA       NA
info.little     5.313e+01  7.098e+05       0        1
info.some       5.313e+01  3.913e+05       0        1
paris.little   -4.438e-06  2.945e+05       0        1
imp.little     -4.402e-06  1.843e+05       0        1
imp.some        5.313e+01  2.783e+05       0        1
safe.li

### Step R2.1 - Bootstrap resampling: sample size sensitivity

In [20]:
dfXY_real <- data.frame(Y = df$y.real, X_clean, check.names = FALSE)
fit_glm_real <- glm(Y ~ ., data = dfXY_real, family = binomial())

vars_to_check <- c("paris.little", "safe.little")
if (length(vars_to_check) == 0) {
  coefs <- summary(fit_glm_real)$coefficients
  if (nrow(coefs) > 1) {
    z_abs <- abs(coefs[-1, "z value"])
    vars_to_check <- names(sort(z_abs, decreasing = TRUE))[1:min(2, length(z_abs))]
  }
}


x_grid <- c(1, 2, 3, 4, 5, 7, 10)   
B      <- 40                     
z_target <- 1.96
p_req    <- 0.80                   


upsample_and_z <- function(dfXY, x, B = 50, var_name){
  N <- nrow(dfXY)
  z_vals <- rep(NA_real_, B)
  for(b in seq_len(B)){
    idx  <- sample.int(N, size = round(x * N), replace = TRUE)
    fitb <- try(glm(Y ~ ., data = dfXY[idx, , drop = FALSE], family = binomial()),
                silent = TRUE)
    if (inherits(fitb, "try-error")) next
    summ <- summary(fitb)$coefficients
    if (!(var_name %in% rownames(summ))) next
    z_vals[b] <- as.numeric(summ[var_name, "z value"])
  }
  z_vals
}

simulate_power_light <- function(dfXY, vars_to_check, x_grid, B = 50,
                                 z_target = 1.96, p_req = 0.80) {
  out <- list()
  for (v in vars_to_check) {
    tab <- data.frame(x = x_grid, p_signif = NA_real_, median_abs_z = NA_real_)
    for (i in seq_along(x_grid)) {
      z_vals <- upsample_and_z(dfXY, x = x_grid[i], B = B, var_name = v)
      z_abs  <- abs(z_vals[is.finite(z_vals)])
      if (length(z_abs) == 0L) next
      tab$median_abs_z[i] <- median(z_abs, na.rm = TRUE)
      tab$p_signif[i]     <- mean(z_abs >= z_target, na.rm = TRUE)
      ## early stop: se già superi p_req, smetti di scalare per questa variabile
      if (!is.na(tab$p_signif[i]) && tab$p_signif[i] >= p_req) break
    }
    idx <- which(!is.na(tab$p_signif) & tab$p_signif >= p_req)
    x_star <- if (length(idx)) min(tab$x[idx]) else NA_real_
    out[[v]] <- data.frame(
      variable = v,
      x_star   = x_star,
      N_star   = ifelse(is.na(x_star), NA_integer_, ceiling(x_star * nrow(dfXY))),
      median_abs_z_at_x_star =
        if (!is.na(x_star)) tab$median_abs_z[match(x_star, tab$x)] else NA_real_,
      p_signif_at_x_star  =
        if (!is.na(x_star)) tab$p_signif[match(x_star, tab$x)] else NA_real_
    )
  }
  do.call(rbind, out)
}

set.seed(123)
cat("\n=== Bootstrap upsampling LIGHT — y.real ===\n")
summary_light_real <- simulate_power_light(dfXY_real, vars_to_check, x_grid, B, z_target, p_req)
print(summary_light_real, row.names = FALSE)


=== Bootstrap upsampling LIGHT — y.real ===


Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non converge"
Warning message:
"glm.fit: l'algoritmo non con

     variable x_star N_star median_abs_z_at_x_star p_signif_at_x_star
 paris.little     NA     NA                     NA                 NA
  safe.little     NA     NA                     NA                 NA


### Step R3.0 - Firth Logistic Regression on dataset 1


In [8]:
### Load package
suppressPackageStartupMessages({
  suppressWarnings({
    library(logistf)
  })
})

### Build datasets
df_yipo  <- data.frame(y.ipo  = df$y.ipo,  X_clean, check.names = FALSE)
df_yreal <- data.frame(y.real = df$y.real, X_clean, check.names = FALSE)

### Fit Firth logistic models
fit_firth_yipo  <- logistf(y.ipo  ~ ., data = df_yipo)
fit_firth_yreal <- logistf(y.real ~ ., data = df_yreal)

cat("\nFirth logistic regression models have been fitted (y.ipo and y.real).\n")



Firth logistic regression models have been fitted (y.ipo and y.real).


### Step R3.1 - Metrics of the model (Firth Logistic)


In [9]:
### In-sample metrics for a fitted Firth model
metrics_firth_insample <- function(fit, y, label) {
  ### Predicted probabilities (in-sample)
  prob <- suppressWarnings(predict(fit, type = "response"))
  
  ### ROC / AUC
  roc_obj <- suppressMessages(pROC::roc(y, prob, quiet = TRUE))
  auc_val <- as.numeric(pROC::auc(roc_obj))
  
  ### Brier score and Tjur's R^2
  brier <- mean((y - prob)^2)
  tjur  <- mean(prob[y == 1]) - mean(prob[y == 0])
  
  ### Pretty print
  cat("\n=== In-sample Metrics (Firth) —", label, "===\n")
  cat("AUC:",   round(auc_val, 3), "\n")
  cat("Brier:", round(brier, 3),   "\n")
  cat("Tjur R²:", round(tjur, 3),  "\n")
}

### Compute metrics using fitted models (from previous step)
metrics_firth_insample(fit_firth_yipo,  df$y.ipo,  "y.ipo")
metrics_firth_insample(fit_firth_yreal, df$y.real, "y.real")



=== In-sample Metrics (Firth) — y.ipo ===
AUC: 1 
Brier: 0.005 
Tjur R²: 0.867 

=== In-sample Metrics (Firth) — y.real ===
AUC: 1 
Brier: 0.005 
Tjur R²: 0.863 


In [10]:
suppressPackageStartupMessages({
  library(pROC)
  })

set.seed(123)

### Stratified k-fold CV for Firth logistic regression
cv_firth <- function(y, X, label, k = 5){
  ### Prepare factors and numeric 0/1
  y_fac <- factor(y, levels = c(0,1))
  y_num <- as.integer(y_fac) - 1

  ### Adjust k to minority class size
  min_class <- min(table(y_fac))
  if (min_class < 2) {
    cat("\n=== CV —", label, "===\nImpossible: minority class < 2 cases.\n")
    return(invisible())
  }
  k_use <- max(2, min(k, min_class))

  ### Create stratified folds
  folds <- createFolds(y_fac, k = k_use, list = TRUE, returnTrain = FALSE)

  aucs <- briers <- tjurs <- rep(NA_real_, length(folds))

  for(i in seq_along(folds)){
    test_idx  <- folds[[i]]
    train_idx <- setdiff(seq_len(length(y_num)), test_idx)

    df_train <- data.frame(y = y_num[train_idx], X[train_idx, , drop = FALSE], check.names = FALSE)
    df_test  <- data.frame(         X[test_idx , , drop = FALSE], check.names = FALSE)
    y_test   <- y_num[test_idx]

    ### Fit Firth on train, predict on test (suppress warnings)
    fit  <- suppressWarnings(logistf(y ~ ., data = df_train))
    prob <- suppressWarnings(predict(fit, newdata = df_test, type = "response"))

    ### Brier (always defined)
    briers[i] <- mean((y_test - prob)^2)

    ### Tjur and AUC only if both classes appear in test
    if (length(unique(y_test)) == 2) {
      tjurs[i] <- mean(prob[y_test == 1]) - mean(prob[y_test == 0])
      roc_obj  <- suppressMessages(roc(y_test, prob, quiet = TRUE))
      aucs[i]  <- as.numeric(auc(roc_obj))
    }
  }

  ### Pretty print
  cat("\n=== ", k_use, "-fold CV — ", label, " ===\n", sep = "")
  fmt <- function(v) if (all(is.na(v))) "NA (a fold had a single class)" else sprintf("%.3f ± %.3f", mean(v, na.rm=TRUE), sd(v, na.rm=TRUE))
  cat("AUC   : ", fmt(aucs),  "\n", sep = "")
  cat("Brier : ", sprintf("%.3f ± %.3f", mean(briers, na.rm=TRUE), sd(briers, na.rm=TRUE)), "\n", sep = "")
  cat("Tjur R²: ", fmt(tjurs), "\n", sep = "")
  if (any(is.na(aucs))) cat("(Note: at least one test fold had a single class)\n")
}

### Run CV on cleaned regressors (X_clean) for both targets
cv_firth(df$y.ipo,  X_clean, "y.ipo",  k = 5)
cv_firth(df$y.real, X_clean, "y.real", k = 5)


Warning message:
"il pacchetto 'pROC' è stato creato con R versione 4.4.3"



=== 5-fold CV — y.ipo ===
AUC   : 1.000 ± 0.000
Brier : 0.010 ± 0.002
Tjur R²: 0.808 ± 0.018

=== 5-fold CV — y.real ===
AUC   : 1.000 ± 0.000
Brier : 0.010 ± 0.003
Tjur R²: 0.817 ± 0.013


### Step R3.2 - Firth Logistic: OR and average marginal effects


In [11]:
### Build datasets (X_clean must contain ONLY regressors)
df_yipo  <- data.frame(y.ipo  = df$y.ipo,  X_clean, check.names = FALSE)
df_yreal <- data.frame(y.real = df$y.real, X_clean, check.names = FALSE)

### Fit Firth logistic models
fit_firth_yipo  <- logistf(y.ipo  ~ ., data = df_yipo)
fit_firth_yreal <- logistf(y.real ~ ., data = df_yreal)

### Tidy function to extract OR / 95% CI / p (exclude intercept)
tidy_logistf <- function(fit){
  est <- coef(fit)
  ci  <- confint(fit)

  ### Robust p-value extraction across versions
  pv <- tryCatch({
    s <- as.data.frame(summary(fit)$coefficients)
    pcol <- grep("(?i)pr|prob|p", names(s), value = TRUE)[1]
    p <- s[[pcol]]; names(p) <- rownames(s); p[names(est)]
  }, error = function(e){
    if (!is.null(fit$prob)) {
      p <- fit$prob; names(p) <- names(est); p
    } else {
      rep(NA_real_, length(est))
    }
  })

  out <- data.frame(
    term    = names(est),
    OR      = exp(est),
    CI_low  = exp(ci[, 1]),
    CI_high = exp(ci[, 2]),
    p_value = as.numeric(pv),
    check.names = FALSE
  )

  out <- out[out$term != "(Intercept)", , drop = FALSE]
  rownames(out) <- NULL
  out[order(out$p_value), , drop = FALSE]
}

### Pretty printer
print_table <- function(tab, label){
  if (nrow(tab) == 0) {
    cat("\n=== Firth —", label, "===\n(No coefficients beyond the intercept)\n")
    return(invisible())
  }
  tab$OR      <- round(tab$OR, 3)
  tab$CI_low  <- round(tab$CI_low, 3)
  tab$CI_high <- round(tab$CI_high, 3)
  tab$p_value <- signif(tab$p_value, 3)
  cat("\n=== Firth —", label, "(OR, 95% CI, p) ===\n")
  print(tab, row.names = FALSE)
}

### Generate and print tables
options(scipen = 999)
tab_ipo  <- tidy_logistf(fit_firth_yipo)
tab_real <- tidy_logistf(fit_firth_yreal)

print_table(tab_ipo,  "y.ipo")
print_table(tab_real, "y.real")


logistf(formula = y.ipo ~ ., data = df_yipo)

Model fitted by Penalized ML
Coefficients:
                      coef  se(coef)  lower 0.95 upper 0.95        Chisq
(Intercept)     -2.3978953  1.477098  -7.2751833 -0.2215505  4.875734370
perc.inv.51.75   6.8237353  6.224887  -6.2390566 20.7550647  1.188356637
perc.reg.lt.25  11.5859092  5.478094  -0.4623755 23.9747284  3.616245935
perc.reg.26.50   5.1059455  2.077294   1.8719637 10.9056889 11.938728596
perc.reg.51.75  23.4574322  9.894963   0.6637177 46.6801673  4.004816555
env.little     -11.9474611  9.013369 -31.3764322 10.0899236  1.472901749
env.some       -17.8502766 11.495373 -43.2039028  9.8518426  1.958303035
env.very         6.9907894  5.866307  -5.6980723 19.8483158  1.348289322
info.little      6.4895792  7.460884 -10.8137101 22.3638781  0.713944081
info.some        5.1573425  4.137618  -5.5057826 13.6785015  1.258696015
paris.little     0.2267733  2.936721  -6.3761575  6.8292699  0.005960327
imp.little       0.1251631  2.06179

In [ ]:
ame_logistf <- function(fit, data){
  ### Build model frame/matrix consistent with the fitted model
  mf <- model.frame(formula(fit), data = data, na.action = na.exclude)
  X  <- model.matrix(attr(mf, "terms"), data = mf)

  ### Align coefficients to design matrix
  beta <- coef(fit)
  keep <- intersect(colnames(X), names(beta))
  X    <- X[, keep, drop = FALSE]
  beta <- beta[keep]

  ### Baseline predictions
  eta <- as.numeric(X %*% beta)
  p   <- suppressWarnings(plogis(eta))

  ### Variables excluding intercept
  vars <- setdiff(colnames(X), "(Intercept)")
  if (length(vars) == 0L) {
    return(data.frame(term = character(), AME_pp = numeric(), type = character()))
  }

  ### Helper: detect 0/1 columns
  is_binary_col <- function(x) {
    ux <- unique(x)
    all(ux %in% c(0,1)) && length(ux) <= 2
  }

  out_list <- lapply(vars, function(v){
    j  <- match(v, colnames(X))
    xj <- X[, j]

    ### Skip NA beta or constant column
    if (is.na(beta[j]) || length(unique(xj)) == 1L) return(NULL)

    if (is_binary_col(xj)) {
      ### Binary AME: difference in predicted prob when flipping xj 0→1
      X1 <- X; X1[, j] <- 1
      X0 <- X; X0[, j] <- 0
      p1 <- suppressWarnings(plogis(as.numeric(X1 %*% beta)))
      p0 <- suppressWarnings(plogis(as.numeric(X0 %*% beta)))
      ame <- mean(p1 - p0, na.rm = TRUE)
      typ <- "binary(0/1)"
    } else {
      ### Continuous AME: mean_i[ p_i*(1-p_i)*beta_j ]
      ame <- mean(p * (1 - p), na.rm = TRUE) * beta[j]
      typ <- "continuous"
    }

    data.frame(term = v, AME_pp = 100 * ame, type = typ, row.names = NULL)
  })

  out <- do.call(rbind, out_list)
  if (is.null(out)) {
    out <- data.frame(term = character(), AME_pp = numeric(), type = character())
  } else {
    out <- out[order(-abs(out$AME_pp)), , drop = FALSE]
    out$AME_pp <- round(out$AME_pp, 3)
  }
  out
}

### Run on your fitted models (fit_firth_yipo / fit_firth_yreal, df_yipo / df_yreal)
options(scipen = 999)

ame_ipo  <- suppressWarnings(ame_logistf(fit_firth_yipo,  df_yipo))
ame_real <- suppressWarnings(ame_logistf(fit_firth_yreal, df_yreal))

cat("\n=== AME (pp) — y.ipo ===\n")
if (nrow(ame_ipo)) print(ame_ipo, row.names = FALSE) else cat("No regressors available.\n")

cat("\n=== AME (pp) — y.real ===\n")
if (nrow(ame_real)) print(ame_real, row.names = FALSE) else cat("No regressors available.\n")



=== AME (pp) — y.ipo ===
           term  AME_pp        type
       env.some -58.486 binary(0/1)
      safe.some -58.317 binary(0/1)
    safe.little  55.732 binary(0/1)
 perc.reg.lt.25  53.369 binary(0/1)
 perc.reg.51.75  51.165 binary(0/1)
       imp.some  50.421 binary(0/1)
       env.very  46.209 binary(0/1)
      info.some  45.287 binary(0/1)
 perc.reg.26.50  44.560 binary(0/1)
     env.little -39.233 binary(0/1)
 perc.inv.51.75  38.911 binary(0/1)
    info.little  38.672 binary(0/1)
   paris.little   1.375 binary(0/1)
     imp.little   0.749 binary(0/1)

=== AME (pp) — y.real ===
           term  AME_pp        type
       env.very  77.512 binary(0/1)
    info.little -12.578 binary(0/1)
   paris.little   8.138 binary(0/1)
    safe.little  -4.133 binary(0/1)
 perc.reg.51.75   3.272 binary(0/1)
 perc.inv.51.75  -3.183 binary(0/1)
      safe.some   2.843 binary(0/1)
 perc.reg.lt.25   2.484 binary(0/1)
       imp.some  -2.484 binary(0/1)
      info.some  -2.040 binary(0/1)
 perc.reg.2

### Step R4.0 - Attempt to clean furthermore the dataset 1

In [13]:
### Parameters
threshold_pct <- 10     ### rarity threshold in %
families <- c("perc.reg\\.", "perc.inv\\.", "env\\.", "info\\.", "safe\\.", "imp\\.")

### Use only the regressors of interest (already in X_clean)
X_chk <- X_clean

### Helper: is 0/1 dummy?
is_dummy01 <- function(x) {
  ux <- unique(na.omit(x))
  length(ux) <= 2 && all(ux %in% c(0, 1))
}

### Select columns matching the family patterns
fam_mask <- Reduce(`|`, lapply(families, function(p) grepl(p, colnames(X_chk), perl = TRUE)))
X_fam <- X_chk[, fam_mask, drop = FALSE]

if (ncol(X_fam) == 0L) {
  cat("\n[NOTICE] No columns found in the expected families.\n")
} else {
  ### Build frequency table
  out_list <- lapply(colnames(X_fam), function(v){
    x <- X_fam[[v]]
    if (!is_dummy01(x)) return(NULL)
    n   <- sum(!is.na(x))
    n1  <- sum(x == 1, na.rm = TRUE)
    n0  <- sum(x == 0, na.rm = TRUE)
    p1  <- if (n > 0) 100 * n1 / n else NA_real_
    fam <- sub("^((perc\\.reg|perc\\.inv|env|info|safe|imp))\\..*$", "\\1", v, perl = TRUE)
    data.frame(
      family   = fam,
      variable = v,
      n        = n,
      ones     = n1,
      zeros    = n0,
      pct_ones = p1,
      rare     = !is.na(p1) && p1 <= threshold_pct,
      stringsAsFactors = FALSE
    )
  })
  freq_tab <- do.call(rbind, out_list)

  if (is.null(freq_tab) || nrow(freq_tab) == 0L) {
    cat("\n[NOTICE] No 0/1 dummies found in the expected families.\n")
  } else {
    ### Order by family and ascending % of 1s
    ord <- order(freq_tab$family, freq_tab$pct_ones, freq_tab$variable)
    freq_tab <- freq_tab[ord, , drop = FALSE]
    rownames(freq_tab) <- NULL

    ### Print summary table
    cat("\n=== Dummy frequencies by family (rare threshold <=", threshold_pct, "%) ===\n")
    freq_tab_print <- freq_tab
    freq_tab_print$pct_ones <- round(freq_tab_print$pct_ones, 2)
    print(freq_tab_print, row.names = FALSE)

    ### Compact list by family: which are rare
    cat("\n=== RARE dummies (≤", threshold_pct, "%) by family ===\n")
    rare_mask <- freq_tab$rare
    if (any(rare_mask)) {
      by_fam <- split(freq_tab[rare_mask, , drop = FALSE], freq_tab$family[rare_mask])
      for (fam in names(by_fam)) {
        vars <- by_fam[[fam]]$variable
        cat("*", fam, "→", paste(vars, collapse = ", "), "\n")
      }
    } else {
      cat("(No rare dummies at the current threshold)\n")
    }
  }
}



=== Dummy frequencies by family (rare threshold <= 10 %) ===
   family       variable  n ones zeros pct_ones  rare
      env       env.very 97   12    85    12.37 FALSE
      env       env.some 97   18    79    18.56 FALSE
      env     env.little 97   62    35    63.92 FALSE
      imp       imp.some 97   11    86    11.34 FALSE
      imp     imp.little 97   38    59    39.18 FALSE
     info      info.some 97    5    92     5.15  TRUE
     info    info.little 97   60    37    61.86 FALSE
 perc.inv perc.inv.51.75 97    6    91     6.19  TRUE
 perc.reg perc.reg.51.75 97    7    90     7.22  TRUE
 perc.reg perc.reg.lt.25 97   38    59    39.18 FALSE
 perc.reg perc.reg.26.50 97   42    55    43.30 FALSE
     safe    safe.little 97   32    65    32.99 FALSE
     safe      safe.some 97   40    57    41.24 FALSE

=== RARE dummies (≤ 10 %) by family ===
* info → info.some 
* perc.inv → perc.inv.51.75 
* perc.reg → perc.reg.51.75 


In [14]:
head(X_clean, 2)

,perc.inv.51.75,perc.reg.lt.25,perc.reg.26.50,perc.reg.51.75,env.little,env.some,env.very,info.little,info.some,paris.little,imp.little,imp.some,safe.little,safe.some
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,1,0,1,0,1,0,0,1,0,1,1,0,0,1
2,0,0,0,0,0,0,1,0,0,0,0,0,0,1


In [15]:
### 1) info.some → merge into lower level: info.little
if ("info.some" %in% colnames(X_clean) && "info.little" %in% colnames(X_clean)) {
  X_clean[["info.little"]] <- as.integer((X_clean[["info.little"]] == 1L) | (X_clean[["info.some"]] == 1L))
  X_clean <- X_clean[, setdiff(colnames(X_clean), "info.some"), drop = FALSE]
  cat("[OK] info.some merged into info.little (kept lower level)\n")
} else {
  cat("[SKIP] info.some→info.little: one of the columns is missing.\n")
}

### 2) perc.reg.51.75 → merge into lower level: perc.reg.26.50
if ("perc.reg.51.75" %in% colnames(X_clean) && "perc.reg.26.50" %in% colnames(X_clean)) {
  X_clean[["perc.reg.26.50"]] <- as.integer((X_clean[["perc.reg.26.50"]] == 1L) | (X_clean[["perc.reg.51.75"]] == 1L))
  X_clean <- X_clean[, setdiff(colnames(X_clean), "perc.reg.51.75"), drop = FALSE]
  cat("[OK] perc.reg.51.75 merged into perc.reg.26.50 (kept lower level)\n")
} else {
  cat("[SKIP] perc.reg.51.75→perc.reg.26.50: one of the columns is missing.\n")
}

### 3) perc.inv.51.75 → unchanged (no lower/upper available)
if ("perc.inv.51.75" %in% colnames(X_clean)) {
  cat("[NOTE] perc.inv.51.75 left unchanged (no suitable merge target)\n")
} else {
  cat("[SKIP] perc.inv.51.75 not present: nothing to do.\n")
}

### (Optional) quick frequency check after merges
post_cols <- intersect(c("info.little","perc.reg.26.50","perc.inv.51.75"), colnames(X_clean))
if (length(post_cols)) {
  cat("\n=== Post-merge frequencies ===\n")
  for (v in post_cols) {
    n1 <- sum(X_clean[[v]] == 1, na.rm = TRUE)
    cat(sprintf("%-16s : %d of %d (%.2f%%)\n", v, n1, nrow(X_clean), 100 * n1 / nrow(X_clean)))
  }
}


[OK] info.some merged into info.little (kept lower level)
[OK] perc.reg.51.75 merged into perc.reg.26.50 (kept lower level)
[NOTE] perc.inv.51.75 left unchanged (no suitable merge target)

=== Post-merge frequencies ===
info.little      : 65 of 97 (67.01%)
perc.reg.26.50   : 49 of 97 (50.52%)
perc.inv.51.75   : 6 of 97 (6.19%)


### Step R5.0 - Firth Logistic Regression on dataset 2

In [16]:
### Rebuild working datasets with grouped regressors
df_yipo  <- data.frame(y.ipo  = df$y.ipo,  X_clean, check.names = FALSE)
df_yreal <- data.frame(y.real = df$y.real, X_clean, check.names = FALSE)

### Fit Firth logistic models (quiet)
fit_firth_yipo  <- suppressWarnings(logistf(y.ipo  ~ ., data = df_yipo))
fit_firth_yreal <- suppressWarnings(logistf(y.real ~ ., data = df_yreal))

cat("\nFirth models fitted on grouped dataset (y.ipo and y.real).\n")



Firth models fitted on grouped dataset (y.ipo and y.real).


### Step R5.1 - Metrics of the model (Firth Logistic)

In [17]:
suppressPackageStartupMessages({
  library(logistf)
  library(pROC)
  library(caret)
})

## --- 1) METRICHE IN-SAMPLE PER MODELLI FIRTH GIÀ FITTATI ---
metrics_firth_insample <- function(fit, y, label) {
  # Probabilità predette in-sample
  prob <- suppressWarnings(predict(fit, type = "response"))

  # ROC / AUC
  roc_obj <- suppressMessages(pROC::roc(y, prob, quiet = TRUE))
  auc_val <- as.numeric(pROC::auc(roc_obj))

  # Brier score e R^2 di Tjur
  brier <- mean((y - prob)^2)
  tjur  <- mean(prob[y == 1]) - mean(prob[y == 0])

  # Stampa ordinata
  cat("\n=== In-sample Metrics (Firth) —", label, "===\n")
  cat("AUC     :", round(auc_val, 3), "\n")
  cat("Brier   :", round(brier, 3),   "\n")
  cat("Tjur R² :", round(tjur, 3),    "\n")
}


## Metriche in-sample per i due target
metrics_firth_insample(fit_firth_yipo,  df$y.ipo,  "y.ipo")
metrics_firth_insample(fit_firth_yreal, df$y.real, "y.real")


## --- 2) STRATIFIED K-FOLD CV PER FIRTH LOGIT ---
set.seed(123)

cv_firth <- function(y, X, label, k = 5) {
  # y come fattore 0/1 e come numerico 0/1
  y_fac <- factor(y, levels = c(0, 1))
  y_num <- as.integer(y_fac) - 1

  # Adatta k alla classe minoritaria
  min_class <- min(table(y_fac))
  if (min_class < 2) {
    cat("\n=== CV —", label, "===\nImpossibile: classe minoritaria < 2 casi.\n")
    return(invisible())
  }
  k_use <- max(2, min(k, min_class))

  # Folds stratificati
  folds <- caret::createFolds(y_fac, k = k_use, list = TRUE, returnTrain = FALSE)

  aucs   <- rep(NA_real_, length(folds))
  briers <- rep(NA_real_, length(folds))
  tjurs  <- rep(NA_real_, length(folds))

  n <- length(y_num)
  idx_all <- seq_len(n)

  for (i in seq_along(folds)) {
    test_idx  <- folds[[i]]
    train_idx <- setdiff(idx_all, test_idx)

    df_train <- data.frame(y = y_num[train_idx], X[train_idx, , drop = FALSE], check.names = FALSE)
    df_test  <- data.frame(         X[test_idx , , drop = FALSE], check.names = FALSE)
    y_test   <- y_num[test_idx]

    # Fit Firth sul train e predici sul test
    fit  <- suppressWarnings(logistf(y ~ ., data = df_train))
    prob <- suppressWarnings(predict(fit, newdata = df_test, type = "response"))

    # Brier (sempre definito)
    briers[i] <- mean((y_test - prob)^2)

    # Tjur e AUC solo se entrambe le classi sono nel test
    if (length(unique(y_test)) == 2) {
      tjurs[i] <- mean(prob[y_test == 1]) - mean(prob[y_test == 0])
      roc_obj  <- suppressMessages(pROC::roc(y_test, prob, quiet = TRUE))
      aucs[i]  <- as.numeric(pROC::auc(roc_obj))
    }
  }

  # Stampa riassuntiva
  cat("\n=== ", k_use, "-fold CV — ", label, " ===\n", sep = "")
  fmt <- function(v) if (all(is.na(v))) "NA (almeno un fold con classe singola)" else sprintf("%.3f ± %.3f", mean(v, na.rm = TRUE), sd(v, na.rm = TRUE))
  cat("AUC    : ", fmt(aucs),  "\n", sep = "")
  cat("Brier  : ", sprintf("%.3f ± %.3f", mean(briers, na.rm = TRUE), sd(briers, na.rm = TRUE)), "\n", sep = "")
  cat("Tjur R²: ", fmt(tjurs), "\n", sep = "")
  if (any(is.na(aucs))) cat("(Nota: almeno un fold test aveva una sola classe)\n")
}

## Esegui CV sui regressori puliti (X_clean) per entrambi i target
cv_firth(df$y.ipo,  X_clean, "y.ipo",  k = 5)
cv_firth(df$y.real, X_clean, "y.real", k = 5)



=== In-sample Metrics (Firth) — y.ipo ===
AUC     : 1 
Brier   : 0.005 
Tjur R² : 0.879 

=== In-sample Metrics (Firth) — y.real ===
AUC     : 1 
Brier   : 0.004 
Tjur R² : 0.872 

=== 5-fold CV — y.ipo ===
AUC    : 1.000 ± 0.000
Brier  : 0.010 ± 0.003
Tjur R²: 0.822 ± 0.023

=== 5-fold CV — y.real ===
AUC    : 1.000 ± 0.000
Brier  : 0.008 ± 0.003
Tjur R²: 0.831 ± 0.013


### Step R5.2 - Metriche OR and average marginal effects

In [18]:
### OR + CI + p (requires tidy_logistf and print_table defined earlier)
tab_ipo_acc  <- tidy_logistf(fit_firth_yipo)
tab_real_acc <- tidy_logistf(fit_firth_yreal)

cat("\n=== Firth (grouped) — y.ipo (OR, 95% CI, p) ===\n")
print_table(tab_ipo_acc,  "y.ipo")

cat("\n=== Firth (grouped) — y.real (OR, 95% CI, p) ===\n")
print_table(tab_real_acc, "y.real")

### AME in percentage points (requires ame_logistf defined earlier)
ame_ipo_acc  <- suppressWarnings(ame_logistf(fit_firth_yipo,  df_yipo))
ame_real_acc <- suppressWarnings(ame_logistf(fit_firth_yreal, df_yreal))

cat("\n=== AME (pp) — y.ipo (grouped) ===\n")
if (nrow(ame_ipo_acc)) {
  print(ame_ipo_acc, row.names = FALSE)
} else {
  cat("No regressors available.\n")
}

cat("\n=== AME (pp) — y.real (grouped) ===\n")
if (nrow(ame_real_acc)) {
  print(ame_real_acc, row.names = FALSE)
} else {
  cat("No regressors available.\n")
}


logistf(formula = y.ipo ~ ., data = df_yipo)

Model fitted by Penalized ML
Coefficients:
                     coef se(coef) lower 0.95 upper 0.95        Chisq
(Intercept)    -2.3978953 1.477098  -7.275183 -0.2215505  4.875734370
perc.inv.51.75  0.4999223 2.784462  -5.718437 15.3395463  0.030420206
perc.reg.lt.25  2.9040073 2.456632  -2.163221 10.3946844  1.222578541
perc.reg.26.50  5.1059455 2.077294   1.871964 10.9056889 11.938728596
env.little      5.8465959 4.163792  -7.965230 27.8735470  0.932063277
env.some        4.3786682 4.058265  -6.586630 14.0392492  0.976028990
env.very        0.2522893 2.726848  -5.949414  7.6392262  0.006596652
info.little    -4.2066615 2.927864 -22.753127  6.3799150  1.983763799
paris.little   -2.4128748 1.948808  -8.387211  4.8608980  1.399594442
imp.little     -4.7476921 1.306794 -13.786371 -2.3050316  2.963361330
imp.some        2.4598640 2.829969  -9.841858 11.5765007  0.590805311
safe.little     1.2275945 1.797277  -9.200865 11.0848429  0.430932974
s

### Step R6.0 - Test a blocchi per dataset 2 su y.real, e ΔAUC/ΔBrier/ΔTjur in CV

In [19]:
### Family regex prefixes (grouped dataset already in X_clean)
families <- c("env\\.", "safe\\.", "info\\.", "imp\\.", "perc\\.reg\\.", "perc\\.inv\\.")

### Build target + regressors frame
dfXY <- data.frame(Y = df$y.real, X_clean, check.names = FALSE)

### Helpers: fitters
fit_glm_full <- function(dfXY)  suppressWarnings(glm(Y ~ ., data = dfXY, family = binomial()))
fit_glm_null <- function(dfXY)  suppressWarnings(glm(Y ~ 1, data = dfXY, family = binomial()))

### Helper: drop a whole family of regressors
drop_family_df <- function(dfXY, family_pat){
  keep <- !grepl(family_pat, colnames(dfXY), perl = TRUE)
  keep[1] <- TRUE  ### always keep Y
  dfXY[, keep, drop = FALSE]
}

### Likelihood Ratio Test between reduced and full models
lrt_glm <- function(reduced, full){
  a <- suppressWarnings(anova(reduced, full, test = "LRT"))
  rr <- a[nrow(a), ]
  c(Chisq = as.numeric(rr$Deviance), df = as.numeric(rr$Df), p = as.numeric(rr$`Pr(>Chi)`))
}

### Run global test and block-wise tests
cat("\n==============================\n")
cat("Target: y.real — glm logit (grouped dataset)\n")
cat("==============================\n")

fit0  <- fit_glm_null(dfXY)
fit1  <- fit_glm_full(dfXY)
lrt_g <- lrt_glm(fit0, fit1)
cat(sprintf("Global LRT (null vs full): ChiSq=%.4f, df=%d, p=%.4g\n",
            lrt_g["Chisq"], as.integer(lrt_g["df"]), lrt_g["p"]))

### Block-wise LRT (no extra metrics)
res <- lapply(families, function(pat){
  dfR <- drop_family_df(dfXY, pat)
  if (ncol(dfR) > 1) {
    fitR <- fit_glm_full(dfR)
    lrt  <- lrt_glm(fitR, fit1)
  } else {
    lrt <- c(Chisq = NA, df = NA, p = NA)
  }
  data.frame(
    family_regex = pat,
    LRT_ChiSq    = as.numeric(lrt["Chisq"]),
    LRT_df       = as.numeric(lrt["df"]),
    LRT_p        = as.numeric(lrt["p"]),
    row.names    = NULL
  )
})

out <- do.call(rbind, res)
out <- out[order(out$LRT_p), , drop = FALSE]
print(out, row.names = FALSE, digits = 4)



Target: y.real — glm logit (grouped dataset)
Global LRT (null vs full): ChiSq=72.6055, df=12, p=1.039e-10
  family_regex       LRT_ChiSq LRT_df LRT_p
        env\\. 0.0000000003874      3     1
       safe\\. 0.0000000000000      2     1
       info\\. 0.0000000000000      1     1
        imp\\. 0.0000000000000      2     1
 perc\\.reg\\. 0.0000000000000      2     1
 perc\\.inv\\. 0.0000000000000      1     1
